# Reproduce PulseLM Methods with WESAD Dataset

1. [PulseLM](https://github.com/manhph2211/PulseLM/)
2. [WESAD](https://archive.ics.uci.edu/dataset/465/wesad+wearable+stress+and+affect+detection)

In [1]:
import os
import sys
import glob

import numpy as np
import pandas as pd

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../python/lamina/'))

# import log_files
import ppg
from data_processing import DataProcessing

## Load Data

- raw wrist BVP/PPG at 64 Hz via the Empatica E4

In [2]:
# def save_all_wrist_data_per_subject(notebook_dir: str):
#     base_dir = DataProcessing.load_base_data_path(notebook_dir)
#     pkl_files = sorted(glob.glob(os.path.join(base_dir, 'WESAD', 'per_subject', 'S*', 'S*.pkl')))

#     all_subject_dfs = []
    
#     for pkl_path in pkl_files:
#         with open(pkl_path, 'rb') as f:
#             data = pd.read_pickle(f)

#         subject = data['subject']
#         wrist = data['signal']['wrist']
#         label_700hz = data['label'].ravel()  # Chest-level 700 Hz ground-truth labels[cite: 1]

#         # Extract raw arrays for wrist channels
#         acc = wrist['ACC']           # 32 Hz[cite: 1]
#         bvp = wrist['BVP'].ravel()   # 64 Hz[cite: 1]
#         eda = wrist['EDA'].ravel()   # 4 Hz[cite: 1]
#         temp = wrist['TEMP'].ravel() # 4 Hz[cite: 1]

#         # Create separate DataFrames per channel
#         df_acc = pd.DataFrame({'ACC_x': acc[:, 0], 'ACC_y': acc[:, 1], 'ACC_z': acc[:, 2]})
#         df_bvp = pd.DataFrame({'BVP': bvp})
#         df_eda = pd.DataFrame({'EDA': eda})
#         df_temp = pd.DataFrame({'TEMP': temp})

#         # Concatenate wrist channels column-wise
#         subj_df = pd.concat([df_acc, df_bvp, df_eda, df_temp], axis=1)

#         # Downsample/align the 700 Hz labels to match wrist DataFrame length
#         label_indices = np.round(np.linspace(0, len(label_700hz) - 1, len(subj_df))).astype(int)
#         subj_df['label'] = label_700hz[label_indices]
#         subj_df['subject'] = subject

#         all_subject_dfs.append(subj_df)

#     # Combine all subject DataFrames
#     combined_df = pd.concat(all_subject_dfs, ignore_index=True)

#     # Save output CSV
#     out_path = os.path.join(base_dir, 'WESAD', 'all_subjects', 'wrist-all_labels.csv')
#     os.makedirs(os.path.dirname(out_path), exist_ok=True)
#     combined_df.to_csv(out_path, index=False)
    
#     # print(f"Saved wrist channels with preserved labels to: {out_path}")
#     return combined_df

In [3]:
# df = save_all_wrist_data_per_subject(notebook_dir)
# df

In [4]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
data_path = os.path.join(base_data_path, 'WESAD/all_subjects/wrist-all_labels.csv')
df = DataProcessing.load_from_file(data_path, file_type='csv')
# df.drop(['Unnamed: 0'], axis=1, inplace=True)

In [5]:
df.columns.to_list()

['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP', 'label', 'subject']

In [6]:
df = df.copy()
df

,ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP,label,subject
0,107.0,-105.0,127.0,10.17,0.349215,33.13,0,S10
1,67.0,-52.0,45.0,12.04,0.346656,33.16,0,S10
2,26.0,40.0,0.0,13.01,0.350494,33.16,0,S10
3,52.0,12.0,46.0,13.07,0.336423,33.16,0,S10
4,42.0,20.0,45.0,12.33,0.338981,33.16,0,S10
...,...,...,...,...,...,...,...,...
5559547,NaN,NaN,NaN,28.59,NaN,NaN,0,S9
5559548,NaN,NaN,NaN,29.45,NaN,NaN,0,S9
5559549,NaN,NaN,NaN,29.07,NaN,NaN,0,S9
5559550,NaN,NaN,NaN,27.74,NaN,NaN,0,S9


## Preprocessing

Label: ID of the respective study protocol condition
- 0 = not defined / transient
- 1 = baseline
- 2 = stress
- 3 = amusement
- 4 = meditation
- 5/6/7 = should be ignored in this dataset

### Get Subject of Interest + Health Metric of Interest

In [7]:
def get_per_subject(df):
    per_subject = {}
    entries = df['subject'].unique()
    for entry in entries:
        filt_subject = (df['subject'] == entry)
        subject_df = df[filt_subject]
        subject_df.reset_index(inplace=True)
        per_subject[entry] = subject_df
    return per_subject

In [8]:
per_subject_dfs = get_per_subject(df)
per_subject_dfs

{'S10':          index  ACC_x  ACC_y  ACC_z    BVP       EDA   TEMP  label subject
 0            0  107.0 -105.0  127.0  10.17  0.349215  33.13      0     S10
 1            1   67.0  -52.0   45.0  12.04  0.346656  33.16      0     S10
 2            2   26.0   40.0    0.0  13.01  0.350494  33.16      0     S10
 3            3   52.0   12.0   46.0  13.07  0.336423  33.16      0     S10
 4            4   42.0   20.0   45.0  12.33  0.338981  33.16      0     S10
 ...        ...    ...    ...    ...    ...       ...    ...    ...     ...
 351739  351739    NaN    NaN    NaN  28.01       NaN    NaN      0     S10
 351740  351740    NaN    NaN    NaN  44.41       NaN    NaN      0     S10
 351741  351741    NaN    NaN    NaN  57.59       NaN    NaN      0     S10
 351742  351742    NaN    NaN    NaN  65.93       NaN    NaN      0     S10
 351743  351743    NaN    NaN    NaN  69.86       NaN    NaN      0     S10
 
 [351744 rows x 9 columns],
 'S11':          index  ACC_x  ACC_y  ACC_z    BVP 

In [9]:
s_df = per_subject_dfs['S7']
s_df

,index,ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP,label,subject
0,4540224,127.0,-50.0,127.0,9.57,5.969716,33.25,0,S7
1,4540225,127.0,46.0,-8.0,7.41,6.100200,33.25,0,S7
2,4540226,90.0,24.0,57.0,5.11,6.096362,33.25,0,S7
3,4540227,40.0,19.0,92.0,2.84,6.088686,33.25,0,S7
4,4540228,19.0,5.0,68.0,0.72,6.114272,33.31,0,S7
...,...,...,...,...,...,...,...,...,...
335227,4875451,NaN,NaN,NaN,-29.93,NaN,NaN,0,S7
335228,4875452,NaN,NaN,NaN,-29.80,NaN,NaN,0,S7
335229,4875453,NaN,NaN,NaN,-29.52,NaN,NaN,0,S7
335230,4875454,NaN,NaN,NaN,-28.97,NaN,NaN,0,S7


In [10]:
s_df['label'].value_counts()

label
0    134591
1     75904
4     50560
2     40960
3     23808
7      3393
5      3200
6      2816
Name: count, dtype: int64

In [11]:
s_df = per_subject_dfs['S7']
s_metric_df = s_df.loc[:, ['BVP', 'label']]
s_metric_df

,BVP,label
0,9.57,0
1,7.41,0
2,5.11,0
3,2.84,0
4,0.72,0
...,...,...
335227,-29.93,0
335228,-29.80,0
335229,-29.52,0
335230,-28.97,0


### Separate Contiguous Protocol Runs & Generate Reset Time Components

In [12]:
run_id = (s_metric_df['label'] != s_metric_df['label'].shift()).cumsum()

# Build continuous run DataFrames with reset time
run_dfs = []
for (lbl, run), group_df in s_metric_df.groupby(['label', run_id]):
    group_df = group_df.reset_index(drop=True)
    group_df = DataProcessing.create_time_df(group_df, sampling_rate=64)
    group_df['run_id'] = run
    run_dfs.append(group_df)

s_raw_runs_df = pd.concat(run_dfs, ignore_index=True)
s_raw_runs_df

,BVP,label,Milliseconds,Seconds,Time,run_id
0,9.57,0,0.00,0.00000,0 days 00:00:00,1
1,7.41,0,15.62,0.01562,0 days 00:00:00.020000,1
2,5.11,0,31.25,0.03125,0 days 00:00:00.030000,1
3,2.84,0,46.88,0.04688,0 days 00:00:00.050000,1
4,0.72,0,62.50,0.06250,0 days 00:00:00.060000,1
...,...,...,...,...,...,...
335227,7.28,7,52937.50,52.93750,0 days 00:00:52.940000,8
335228,6.84,7,52953.12,52.95312,0 days 00:00:52.950000,8
335229,6.39,7,52968.75,52.96875,0 days 00:00:52.970000,8
335230,6.01,7,52984.38,52.98438,0 days 00:00:52.980000,8


## Reproducibility Steps

1. Resample to 125 Hz
2. Fourth-order Butterworth low-pass filter at 8 Hz
3. DC offset removal
4. Slice into 10-second windows
5. Per-segment min-max normalization

#### 1. Resample to 125 Hz

#### 2. Fourth-order Butterworth low-pass filter at 8 Hz


#### 3. DC offset removal

#### 4. Slice into 10-second windows

#### 5. Per-segment min-max normalization